## GOSDT (optimal sparse decision tree) — PyPI `gosdt`

Same feature pipeline as elsewhere: `get_preprocessor_with_impute` from `public.model_IAI` with `CAT_COLUMNS`, `TRUE_NUM_COLUMNS`, `BIN_FLAG_COLUMNS`.

The **PyPI** package exposes `GOSDTClassifier` (boolean features, misclassification **cost**). It does not accept the Jimmy-Lin README `"objective": "f1"`.

**Cost modes:** `GOSDT_COST_MODE = "balance"` uses **`balance=True`** and the **library default** class-reweighted cost (do **not** pass `cost_matrix` to `fit`). `"custom"` uses an **F1-style asymmetric** `cost_matrix` with `GOSDT_F1_W` and **`balance=False`**.

**`GOSDT_SAFE_MODE` (default `True`):** uses safe `regularization`, `time_limit`, and `depth_budget`. Set **`GOSDT_SAFE_MODE = False`** for your own hyperparameters (may OOM on full data). **`GOSDT_MAX_TRAIN_ROWS = None`** uses all training rows. Continuous/OHE columns are binarized with **training-set median** splits.

Install: `pip install gosdt`

**Paths:** The first code cell defines `PROJECT_ROOT` (this notebook’s folder when using Cursor/VS Code, otherwise the kernel working directory), loads the parquet from that folder, and adds it to `sys.path` so `public` and `symmetric_excess_AUC` import correctly.

In [2]:
import importlib
import sys
from pathlib import Path

import pandas as pd
import numpy as np

import public.model_IAI
importlib.reload(public.model_IAI)

# Project root: directory of this .ipynb when running in Cursor/VS Code; else kernel cwd.
_ipynb = globals().get("__vsc_ipynb_file__")
if _ipynb:
    PROJECT_ROOT = Path(_ipynb).resolve().parent
else:
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

_parquet_path = PROJECT_ROOT / "0917_2017_18_with_2017_cost.parquet"
if not _parquet_path.is_file():
    raise FileNotFoundError(
        f"Parquet not found: {_parquet_path}. Place the file there or set the kernel cwd to my_projects."
    )
df_og = pd.read_parquet(_parquet_path)

TRAIN_TEST_SEED = 123 #9012

# Fetch helper functions directly from public.model_IAI
get_bin_flag_columns = public.model_IAI.get_bin_flag_columns
get_true_num_columns = public.model_IAI.get_true_num_columns
train_test_split_enrol = public.model_IAI.train_test_split_enrol

BIN_FLAG_COLUMNS = get_bin_flag_columns(df_og) 
STAGE_COLUMNS = [col for col in df_og.columns if "stage" in col.lower()]
CAT_COLUMNS = df_og.select_dtypes(include=["object","category"]).columns.tolist()
TRUE_NUM_COLUMNS = get_true_num_columns(df_og, CAT_COLUMNS, BIN_FLAG_COLUMNS) 

print("categorical cols: ", CAT_COLUMNS)
print("stage cols: ", STAGE_COLUMNS)
print("bin flag cols: ", BIN_FLAG_COLUMNS)
print("true num cols: ", TRUE_NUM_COLUMNS)
leftover_cols = [
    c for c in df_og.columns 
    if c not in CAT_COLUMNS and c not in TRUE_NUM_COLUMNS and c not in STAGE_COLUMNS and c not in BIN_FLAG_COLUMNS 
]

print(f"Number of leftover columns: {len(leftover_cols)}")
print(leftover_cols, df_og.shape)  # preview first 50

def make_cost_stratum_3class(df):
    # Default to low-cost (class 0)
    cost_stratum = pd.Series(0, index=df.index)
    cost_stratum[(df['highcost_gt_50000'] == 1) & (df['highcost_gt_100000'] == 0)] = 1
    # Emergent high cost (class 2): 100k to 200k
    cost_stratum[(df['highcost_gt_100000'] == 1) & (df['highcost_gt_200000'] == 0)] = 2
    # High cost (class 2): 200k+
    cost_stratum[df['highcost_gt_200000'] == 1] = 3
    return cost_stratum

# Add the new column to your data
df_og['cost_stratum_2018'] = make_cost_stratum_3class(df_og)
print(df_og["cost_stratum_2018"].value_counts(dropna=False))

cutoff_columns = [col for col in df_og.columns if col.startswith('highcost_gt_')]

feature_cols = [c for c in df_og.columns
    if c not in  (['annual_cost_2017','annual_cost_2018_deflated',"ENROLID", "cost_stratum_2018"] + cutoff_columns)
]
numeric_cols = df_og[feature_cols + ["cost_stratum_2018"]].select_dtypes(include=["number"]).columns
corrs = df_og[numeric_cols].corr()["cost_stratum_2018"].abs().sort_values(ascending=False)
# Columns to drop
high_corr_cols = corrs[corrs > 0.5].index.tolist()
# Remove the target column itself, if present
high_corr_cols = [col for col in high_corr_cols if col != "cost_stratum_2018"]
# Final filtered feature set
feature_cols = [col for col in feature_cols if col not in high_corr_cols]
print("High corr features dropped from prediction columns: ", high_corr_cols)
target_col = "highcost_gt_200000"

# Split data into train/test/val (same as multiobjective_bilevel.ipynb)
train_ids, test_ids, train_pd, test_pd = train_test_split_enrol(
    df_og,
    target_col="cost_stratum_2018",
    test_size=0.3,
    verbose=False,
    random_state=TRAIN_TEST_SEED
)
print(f"Train shape: {train_pd.shape}, Test shape: {test_pd.shape}")
print("Feature cols:", len(feature_cols))

val_ids, test_ids, val_pd, test_pd = train_test_split_enrol(
    test_pd, 
    target_col=target_col,
    test_size=0.5,
    verbose=False,
    random_state=TRAIN_TEST_SEED
)
X_test = test_pd[feature_cols]
y_test = test_pd[target_col]
X_val = val_pd[feature_cols]
y_val = val_pd[target_col]

print(f"Train: {train_pd.shape}, Val: {val_pd.shape}, Test: {test_pd.shape}")
print(f"Train target distribution:\n{train_pd[target_col].value_counts()}")

FileNotFoundError: Parquet not found: /Users/cat2510/my_projects/0917_2017_18_with_2017_cost.parquet. Place the file there or set the kernel cwd to my_projects.

In [3]:
# GOSDTClassifier: same preprocessor as OCT baselines; F1-aligned cost matrix (PyPI has no literal F1 objective).
import sys
import subprocess

try:
    from gosdt import GOSDTClassifier
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gosdt", "-q"])
    from gosdt import GOSDTClassifier

import numpy as np
from public.model_IAI import get_preprocessor_with_impute

# --- Tuning knobs (small lambda -> huge native search; often OOM / kernel crash on laptop) ---
# SAFE_MODE=True: README-style lambda, short time cap, depth cap -- intended to finish reliably.
# Set SAFE_MODE=False only if you accept crashes or run on a large machine.
GOSDT_SAFE_MODE = True

GOSDT_REGULARIZATION_SAFE = 0.01
GOSDT_TIME_LIMIT_SAFE = 600
GOSDT_DEPTH_BUDGET_SAFE = 4

GOSDT_REGULARIZATION = 1e-4
GOSDT_REGULARIZATION_FLOOR = 0.001
GOSDT_TIME_LIMIT_SEC = 3600
GOSDT_DEPTH_BUDGET = None

GOSDT_MAX_TRAIN_ROWS = None
GOSDT_COST_MODE = "balance"
GOSDT_F1_W = 0.9

X_train_df = train_pd[feature_cols].copy()
y_train_s = train_pd[target_col]
X_val_df = val_pd[feature_cols].copy()
y_val_s = val_pd[target_col]
X_test_df = test_pd[feature_cols].copy()
y_test_s = test_pd[target_col]

gosdt_preprocessor = get_preprocessor_with_impute(
    X_train_df,
    categorical_cols=CAT_COLUMNS,
    numeric_cols=TRUE_NUM_COLUMNS,
    binary_cols=BIN_FLAG_COLUMNS,
    verbose=True,
)
X_tr = gosdt_preprocessor.fit_transform(X_train_df)
X_va = gosdt_preprocessor.transform(X_val_df)
X_te = gosdt_preprocessor.transform(X_test_df)

feat_names = list(gosdt_preprocessor.get_feature_names_out())
thresh = np.median(X_tr, axis=0)
X_tr_bool = np.ascontiguousarray((X_tr > thresh).astype(bool))
X_va_bool = np.ascontiguousarray((X_va > thresh).astype(bool))
X_te_bool = np.ascontiguousarray((X_te > thresh).astype(bool))
del X_tr, X_va, X_te
import gc as _gc
_gc.collect()

y_train_arr = y_train_s.values.astype(np.int64)
y_val_arr = y_val_s.values.astype(np.int64)
y_test_arr = y_test_s.values.astype(np.int64)

_n_full = int(X_tr_bool.shape[0])
if GOSDT_MAX_TRAIN_ROWS is not None and _n_full > int(GOSDT_MAX_TRAIN_ROWS):
    from sklearn.model_selection import train_test_split

    _idx, _ = train_test_split(
        np.arange(_n_full, dtype=np.int64),
        train_size=int(GOSDT_MAX_TRAIN_ROWS),
        stratify=y_train_arr,
        random_state=TRAIN_TEST_SEED,
    )
    X_tr_bool = np.ascontiguousarray(X_tr_bool[_idx])
    y_train_arr = y_train_arr[_idx]
    print(
        f"GOSDT fit uses stratified subsample n={X_tr_bool.shape[0]} "
        f"(GOSDT_MAX_TRAIN_ROWS={GOSDT_MAX_TRAIN_ROWS}); full train rows were {_n_full}."
    )

n_train = y_train_arr.shape[0]
w = float(GOSDT_F1_W)
if GOSDT_COST_MODE not in ("balance", "custom"):
    raise ValueError('GOSDT_COST_MODE must be "balance" or "custom"')
use_balance = GOSDT_COST_MODE == "balance"
cost_matrix = None
if GOSDT_COST_MODE == "custom":
    cost_matrix = np.array(
        [[0.0, w / n_train], [(1.0 - w) / n_train, 0.0]],
        dtype=np.float64,
    )

if GOSDT_SAFE_MODE:
    fit_reg = float(GOSDT_REGULARIZATION_SAFE)
    time_limit = int(GOSDT_TIME_LIMIT_SAFE)
    depth_budget = GOSDT_DEPTH_BUDGET_SAFE
    print(
        "GOSDT_SAFE_MODE=True: using strong regularization (sparse tree), "
        f"short time_limit={time_limit}s, depth_budget={depth_budget}. "
        "Set GOSDT_SAFE_MODE=False for aggressive search (may crash)."
    )
else:
    fit_reg = max(float(GOSDT_REGULARIZATION), float(GOSDT_REGULARIZATION_FLOOR))
    time_limit = int(GOSDT_TIME_LIMIT_SEC)
    depth_budget = GOSDT_DEPTH_BUDGET
    if fit_reg > float(GOSDT_REGULARIZATION):
        print(
            f"Note: effective regularization {fit_reg} (floor {GOSDT_REGULARIZATION_FLOOR}); "
            f"requested was {GOSDT_REGULARIZATION}."
        )

print(
    f"GOSDT_COST_MODE={GOSDT_COST_MODE!r}: "
    + (
        "library default balanced costs (no custom cost_matrix to fit)."
        if use_balance
        else f"custom F1-style cost_matrix with w={w}."
    )
)

gosdt_clf = GOSDTClassifier(
    regularization=fit_reg,
    allow_small_reg=True,
    time_limit=time_limit,
    depth_budget=depth_budget,
    balance=use_balance,
    cancellation=True,
    look_ahead=True,
    similar_support=True,
    verbose=True,
)
print(
    f"Fitting GOSDTClassifier: regularization={fit_reg}, time_limit={time_limit}s, "
    f"depth_budget={depth_budget}, balance={use_balance}, n_train={n_train}, "
    f"n_features_bool={X_tr_bool.shape[1]}"
)
if cost_matrix is None:
    gosdt_clf.fit(X_tr_bool, y_train_arr, input_features=feat_names)
else:
    gosdt_clf.fit(X_tr_bool, y_train_arr, input_features=feat_names, cost_matrix=cost_matrix)
res = gosdt_clf.get_result()
print("GOSDT status:", res["status"], "time (s):", res["time"], "iterations:", res["n_iterations"])


→ Building preprocessor w/ conditional imputation:
   • Cat: OHE on: ['INCOME_LEVEL', 'AGEGRP', 'SEX', 'REGION', 'cost_pattern_2017', 'cost_stability_2017', 'lab_monitoring_intensity']
   • Num: scale on: ['stage_2017', 'util_2017', '2017Q1_ckd_cost', '2017Q1_ckd_claims', '2017Q1_max_ckd_stage', '2017Q1_direct_ckd_cost', '2017Q1_procedure_ckd_cost', '2017Q1_comorbidity_ckd_cost', '2017Q2_ckd_cost', '2017Q2_ckd_claims', '2017Q2_max_ckd_stage', '2017Q2_direct_ckd_cost', '2017Q2_procedure_ckd_cost', '2017Q2_comorbidity_ckd_cost', '2017Q3_ckd_cost', '2017Q3_ckd_claims', '2017Q3_max_ckd_stage', '2017Q3_direct_ckd_cost', '2017Q3_procedure_ckd_cost', '2017Q3_comorbidity_ckd_cost', '2017Q4_ckd_cost', '2017Q4_ckd_claims', '2017Q4_max_ckd_stage', '2017Q4_direct_ckd_cost', '2017Q4_procedure_ckd_cost', '2017Q4_comorbidity_ckd_cost', 'ckd_cost_trend_2017', 'ckd_cost_volatility_2017', 'ckd_cost_deriv_Q1_Q2_2017', 'ckd_cost_deriv_Q2_Q3_2017', 'ckd_cost_deriv_Q3_Q4_2017', 'total_increasing_quarters_20

In [3]:
import json
from pprint import pprint

# 1) Raw JSON tree(s)
res = gosdt_clf.get_result()
models = json.loads(res["models_string"])
print(f"num_models={len(models)}")
pprint(models[0])   # first tree

# 2) Built tree object (compact string)
print(gosdt_clf.trees_[0])

num_models=1
{'false': {'complexity': 0.009999999776482582,
           'loss': 0.05303030088543892,
           'prediction': 0},
 'feature': 56,
 'true': {'false': {'false': {'complexity': 0.009999999776482582,
                              'loss': 0.08484847843647003,
                              'prediction': 0},
                    'feature': 65,
                    'true': {'complexity': 0.009999999776482582,
                             'loss': 0.05254393070936203,
                             'prediction': 1}},
          'feature': 19,
          'true': {'complexity': 0.009999999776482582,
                   'loss': 0.04597046226263046,
                   'prediction': 1}}}
<class 'gosdt._tree.Tree'>: { feature: 56 [ left child: { feature: 19 [ left child: { prediction: 1, loss: 0.04597046226263046 }, right child: { feature: 65 [ left child: { prediction: 1, loss: 0.05254393070936203 }, right child: { prediction: 0, loss: 0.08484847843647003 }] }] }, right child: { prediction: 0

In [4]:
def print_tree(node, feature_names, indent=""):
    if hasattr(node, "prediction"):  # Leaf
        print(f"{indent}Predict {node.prediction} (loss={node.loss:.6g})")
        return
    f = feature_names[node.feature]
    print(f"{indent}if {f} == True:")
    print_tree(node.left_child, feature_names, indent + "  ")
    print(f"{indent}else:")
    print_tree(node.right_child, feature_names, indent + "  ")

t = gosdt_clf.trees_[0]
print_tree(t.tree, t.features)

if num__quarterly_max_2017 == True:
  if num__stage_2017 == True:
    Predict 1 (loss=0.0459705)
  else:
    if binary__has_Acute_Kidney_Failure == True:
      Predict 1 (loss=0.0525439)
    else:
      Predict 0 (loss=0.0848485)
else:
  Predict 0 (loss=0.0530303)


In [12]:
import numpy as np
from sklearn.metrics import (
    precision_recall_fscore_support,
    average_precision_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
)


def specificity_from_cm(cm):
    tn, fp, fn, tp = cm.ravel()
    denom = tn + fp
    return float(tn / denom) if denom > 0 else 0.0


def recall_specificity_gmean_at_best_threshold(y_true, y_score_pos):
    """Maximize geometric mean of sensitivity (recall) and specificity over ROC thresholds."""
    y_true = np.asarray(y_true)
    y_score_pos = np.asarray(y_score_pos)
    fpr, tpr, thr = roc_curve(
        y_true, y_score_pos, pos_label=1, drop_intermediate=False
    )
    spec = 1.0 - fpr
    gmean = np.sqrt(np.clip(tpr * spec, 0.0, None))
    j = int(np.argmax(gmean))
    return {
        "recall": float(tpr[j]),
        "specificity": float(spec[j]),
        "gmean": float(gmean[j]),
        "threshold": float(thr[j]),
    }


y_hat_test = gosdt_clf.predict(X_te_bool)
pr_te, rc_te, f1_te, _ = precision_recall_fscore_support(
    y_test_arr, y_hat_test, average="binary", pos_label=1, zero_division=0
)
cm_te = confusion_matrix(y_test_arr, y_hat_test, labels=[0, 1])
spec_te = specificity_from_cm(cm_te)
# Scores for ROC/PR (positive class probability)
try:
    proba_te = gosdt_clf.predict_proba(X_te_bool)
    if proba_te.shape[1] >= 2:
        pos_scores_te = proba_te[:, 1]
    else:
        pos_scores_te = proba_te.ravel()
    ap_te = average_precision_score(y_test_arr, pos_scores_te)
    try:
        roc_te = roc_auc_score(y_test_arr, pos_scores_te)
    except ValueError:
        roc_te = float("nan")
    gmean_te = recall_specificity_gmean_at_best_threshold(
        y_test_arr, pos_scores_te
    )
except Exception as ex:
    pos_scores_te = None
    ap_te = float("nan")
    roc_te = float("nan")
    gmean_te = None
    print("predict_proba / AP skipped:", ex)
print("\n--- GOSDT test set (hard 0/1 predictions) ---")
print(
    f"precision={pr_te:.4f}  recall={rc_te:.4f}  specificity={spec_te:.4f}  F1={f1_te:.4f}"
)
if np.isfinite(ap_te):
    print(f"average_precision (AP)={ap_te:.4f}")
if np.isfinite(roc_te):
    print(f"ROC-AUC={roc_te:.4f}")
if gmean_te is not None:
    print(
        "At threshold maximizing gmean=sqrt(recall*specificity): "
        f"recall={gmean_te['recall']:.4f}  specificity={gmean_te['specificity']:.4f}  "
        f"gmean={gmean_te['gmean']:.4f}  threshold={gmean_te['threshold']:.6g}"
    )
print("confusion_matrix [ [TN, FP], [FN, TP] ] labels 0,1:\n", cm_te)
print("\n--- GOSDT validation set (for reference) ---")
y_hat_val = gosdt_clf.predict(X_va_bool)
pr_va, rc_va, f1_va, _ = precision_recall_fscore_support(
    y_val_arr, y_hat_val, average="binary", pos_label=1, zero_division=0
)
cm_va = confusion_matrix(y_val_arr, y_hat_val, labels=[0, 1])
spec_va = specificity_from_cm(cm_va)
print(
    f"precision={pr_va:.4f}  recall={rc_va:.4f}  specificity={spec_va:.4f}  F1={f1_va:.4f}"
)
try:
    proba_va = gosdt_clf.predict_proba(X_va_bool)
    if proba_va.shape[1] >= 2:
        pos_scores_va = proba_va[:, 1]
    else:
        pos_scores_va = proba_va.ravel()
    gmean_va = recall_specificity_gmean_at_best_threshold(
        y_val_arr, pos_scores_va
    )
    print(
        "At threshold maximizing gmean: "
        f"recall={gmean_va['recall']:.4f}  specificity={gmean_va['specificity']:.4f}  "
        f"gmean={gmean_va['gmean']:.4f}  threshold={gmean_va['threshold']:.6g}"
    )
except Exception as ex:
    print("gmean / proba on val skipped:", ex)


--- GOSDT test set (hard 0/1 predictions) ---
precision=0.0884  recall=0.6831  specificity=0.7955  F1=0.1566
average_precision (AP)=0.0693
ROC-AUC=0.7393
At threshold maximizing gmean=sqrt(recall*specificity): recall=0.6831  specificity=0.7955  gmean=0.7372  threshold=1
confusion_matrix [ [TN, FP], [FN, TP] ] labels 0,1:
 [[3890 1000]
 [  45   97]]

--- GOSDT validation set (for reference) ---
precision=0.1000  recall=0.7376  specificity=0.8086  F1=0.1761
At threshold maximizing gmean: recall=0.7376  specificity=0.8086  gmean=0.7723  threshold=1


/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/cat2510/.pyenv/versions/3.11.0/envs/analytics/lib/python3.11/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [4]:
from gosdt._tree import Leaf

def gosdt_leaf_ids(X_bool, tree_wrap):
    """Integer leaf id per row; ids 0..L-1 in DFS order over leaf nodes."""
    root = tree_wrap.tree

    def enumerate_leaves(node, leaves=None):
        if leaves is None:
            leaves = []
        if isinstance(node, Leaf):
            leaves.append(node)
            return leaves
        enumerate_leaves(node.left_child, leaves)
        enumerate_leaves(node.right_child, leaves)
        return leaves

    leaves = enumerate_leaves(root)
    leaf_obj_to_id = {id(L): i for i, L in enumerate(leaves)}

    def walk(x, node):
        if isinstance(node, Leaf):
            return leaf_obj_to_id[id(node)]
        if x[node.feature]:
            return walk(x, node.left_child)
        return walk(x, node.right_child)

    X_bool = np.asarray(X_bool, dtype=bool)
    return np.array([walk(X_bool[i], root) for i in range(X_bool.shape[0])], dtype=int)

# After fit:
proba_te = gosdt_clf.predict_proba(X_te_bool)
pos = proba_te[:, 1] if proba_te.shape[1] >= 2 else proba_te.ravel()
leaf_g = gosdt_leaf_ids(X_te_bool, gosdt_clf.trees_[0])

gosdt_pred_df = pd.DataFrame({
    "leaf_assignment": leaf_g,
    "predicted_proba": pos,
})
# If you add ENROLID for alignment:
# gosdt_pred_df.insert(0, "ENROLID", test_pd["ENROLID"].values)

out_path = PROJECT_ROOT / "gosdt_out" / f"seed_{TRAIN_TEST_SEED}" / "predictions" / "gosdt_predictions.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
gosdt_pred_df.to_csv(out_path, index=False)
print(f"Saved GOSDT predictions to {out_path}")

Saved GOSDT predictions to /Users/cat2510/my_projects/gosdt_out/seed_123/predictions/gosdt_predictions.csv


In [5]:
from symmetric_excess_AUC import symmetric_leaf_evaluation_oct

mv_pred_path = PROJECT_ROOT / "vanilla_oct" / f"seed_{TRAIN_TEST_SEED}" / "predictions" / "oct_predictions.csv"
if not mv_pred_path.is_file():
    raise FileNotFoundError(
        f"Vanilla OCT predictions not found: {mv_pred_path}. "
        "Expected path under PROJECT_ROOT (see first cell)."
    )

res = symmetric_leaf_evaluation_oct(
    mv_pred_path=str(mv_pred_path),
    ms_pred_path=str(out_path),
    y_test=y_test_arr,
    enrolid_col=None,
)

In [ ]:
res

In [8]:
ours_path = PROJECT_ROOT / "two_stage_kcenter_results_global" / "predictions" / "oct_predictions_ckd_best_curated_prauc.csv"
ours_path.parent.mkdir(parents=True, exist_ok=True)
mv_pred_path = PROJECT_ROOT / "vanilla_oct" /"predictions" / "oct_predictions.csv"
res = symmetric_leaf_evaluation_oct(
    mv_pred_path=str(mv_pred_path),
    ms_pred_path=str(ours_path),
    y_test=y_test_arr,
    enrolid_col=None,
)

In [9]:
res

{'overall': {'M_v_global_ROC': 0.5908011463463809,
  'M_v_global_PR': 0.11287952462305123,
  'M_v_global_MCC': 0.2543349866095827,
  'M_s_global_ROC': 0.7349995679599066,
  'M_s_global_PR': 0.1443849676152797,
  'M_s_global_MCC': 0.2750127608881647,
  'n_test': 5032,
  'prevalence': 0.02821939586645469},
 'overall_ci': {'global_ROC_Mv': {'point': 0.5912373970855807,
   'lo': 0.5601098584620336,
   'hi': 0.6236686401013485,
   'se_boot': 0.016654048549592265,
   'n_eff': 2000},
  'global_ROC_Ms': {'point': 0.7356269823156574,
   'lo': 0.6788881810762839,
   'hi': 0.78707214634982,
   'se_boot': 0.02756836704336109,
   'n_eff': 2000},
  'global_ROC_diff_Ms_minus_Mv': {'point': 0.14438958523007672,
   'lo': 0.09263845461107884,
   'hi': 0.1926575612161385,
   'se_boot': 0.02541727044906093,
   'n_eff': 2000},
  'global_PR_Mv': {'point': 0.1151554003692086,
   'lo': 0.06938357554320827,
   'hi': 0.17024586018776672,
   'se_boot': 0.025924298855276966,
   'n_eff': 2000},
  'global_PR_Ms': {